# Snow Cover Mapping with VIIRS

This notebook uses data from the Visible Infrared Imaging Radiometer Suite (VIIRS) to create a simple map of snow cover. Because this is NASA data, we can use `earthaccess` to quickly access a few images of interest.

Credit goes to Justin Pflug for the snow cover processing code.

One of the plotting cells requires the `contextily` package. If you do not have this package, then either run the cell below, or change up the plotting code to your liking.

In [ ]:
!pip install contextily

In [ ]:
import earthaccess
import geopandas as gpd
import glob
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
import h5py
import numpy as np
import datetime
import contextily as cx

The VIIRS snow cover data is available through NSIDC, so we are capable of streaming it directly through the cloud. Let's start with a simple example using only a single file.

According to its NSIDC webpage (https://nsidc.org/data/vj110a1f/versions/2), the short name for the VIIRS snow cover product is `VJ110A1F`. We will look at an image that includes Fairbanks, AK during the SnowEx campaign in March 2023.

In [ ]:
# Authenticate with earthaccess
earthaccess.login()

In [ ]:
# Convert Fairbanks geoJSON to earthaccess-friendly format.
fairbanks = gpd.read_file("/home/jovyan/snow-observations-cookbook/notebooks/data/cffl_lidar_box.geojson")
fairbanks.to_crs("EPSG:4326", inplace=True)
minx, miny, maxx, maxy = fairbanks.total_bounds

# Run the earthaccess query
results = earthaccess.search_data(
    short_name='VJ110A1F',
    bounding_box=(minx, miny, maxx, maxy),
    temporal=("2023-03-01", "2023-03-31"),
    count=10
)

In [ ]:
display(results[1])

Since we are looking at a large area over a month, we have several files available for analysis. To start, we will only look at one file, to demonstrate how we would load and plot the snow cover data.

In [ ]:
# Stream the data directly into memory
files = earthaccess.open(results)

VIIRS snow cover data is provided in HDF-5 format, with a customized GIS projection. Neither setup is ideal for a simple plot, so we will start by defining functions that will make data visualizing easier.

In [ ]:
# Function to identify coordinates and snow cover data
def assign_and_grid(ds):
    with h5py.File(files[1],'r') as f:
        x = f['HDFEOS']['GRIDS']['VIIRS_Grid_IMG_2D']['XDim'][:] # X-coordinates
        y = f['HDFEOS']['GRIDS']['VIIRS_Grid_IMG_2D']['YDim'][:] # Y-coordinates
        data = f['HDFEOS']['GRIDS']['VIIRS_Grid_IMG_2D']['Data Fields']['CGF_NDSI_Snow_Cover'][:] # Snow cover

    # Create coordinate arrays (adjust if you need a specific dimension order)
    coords = {'x': x, 'y': y}
    # Create the xarray dataset
    ds = xr.DataArray(data, coords=coords, dims=['y', 'x'])
    # If you need to create a full Dataset (not just a DataArray), you can do:
    ds = ds.to_dataset(name='CGF_NDSI_Snow_Cover')
    return ds

# Ensure that the data has a defined CRS, prior to reprojection
def reproject_func(ds,var,dom_ref,x_lims,y_lims):
    ds = ds[var]
    ds.rio.write_crs("PROJCS[\"MODIS Sinusoidal\",GEOGCS[\"GCS_unnamed ellipse\",DATUM[\"D_unknown\",SPHEROID[\"Unknown\",6371007.181,0]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433]],PROJECTION[\"Sinusoidal\"],PARAMETER[\"false_easting\",0.0],PARAMETER[\"false_northing\",0.0],PARAMETER[\"central_meridian\",0.0],UNIT[\"Meter\",1.0]]", inplace=True)
    #ds = ds.sel(x=slice(x_lims[0],x_lims[1]),y=slice(y_lims[1],y_lims[0]))
    return ds

In [ ]:
# Open the data in raster format, to define domain limits
dom_ref = rxr.open_rasterio(files[0]).isel(band=0).rio.write_crs(4326)

# Define domain limits
buffer = 5000
lim_test = dom_ref.rio.reproject("PROJCS[\"MODIS Sinusoidal\",GEOGCS[\"GCS_unnamed ellipse\",DATUM[\"D_unknown\",SPHEROID[\"Unknown\",6371007.181,0]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433]],PROJECTION[\"Sinusoidal\"],PARAMETER[\"false_easting\",0.0],PARAMETER[\"false_northing\",0.0],PARAMETER[\"central_meridian\",0.0],UNIT[\"Meter\",1.0]]")
x_lims = [lim_test.x.mean().values-buffer,lim_test.x.mean().values+buffer]
y_lims = [lim_test.y.mean().values-buffer,lim_test.y.mean().values+buffer]
print(x_lims,y_lims)

In [ ]:
try:
    # Load and pre-process VIIRS data
    ds = xr.open_dataset(files[0])
    ds = assign_and_grid(ds)
    ds = reproject_func(ds,'CGF_NDSI_Snow_Cover',dom_ref,x_lims,y_lims)
    
    # Filter data to include realistic values
    ds = ds.where(ds <= 100)

    if len(ds.values[~np.isnan(ds.values)]) > 0:
        print(ds.shape)
except:
    print('Error: Unable to process file.')

In [ ]:
# Get date of data granule
date = dom_ref.attrs['StartTime'][:10]

# Plot VIIRS snow cover over ESRI imagery
fig, ax = plt.subplots()
ds.rio.reproject(4326).plot(ax=ax, vmin=0, vmax=100, cmap="Blues_r",
                            cbar_kwargs={'label': "Snow cover fraction [%]"})
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"VIIRS snow cover, {date}")
ax.set_xlim(-180, -140)
ax.set_ylim(60, 68)
cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.Esri.WorldImagery, zoom=8)

## Working with multiple VIIRS files
The above example demonstrates that we can use `earthaccess` to quickly access VIIRS snow cover data, but it only considers one swath data.

If one looks closely to our original query, we actually retrieved 10 files over the domain of interest. Let's see if we can utilize all of those files for a more detailed analysis.

In [ ]:
def assign_and_grid(ds, file):
    with h5py.File(file,'r') as f:
        x = f['HDFEOS']['GRIDS']['VIIRS_Grid_IMG_2D']['XDim'][:]
        y = f['HDFEOS']['GRIDS']['VIIRS_Grid_IMG_2D']['YDim'][:]
        data = f['HDFEOS']['GRIDS']['VIIRS_Grid_IMG_2D']['Data Fields']['CGF_NDSI_Snow_Cover'][:]

    # Create coordinate arrays (adjust if you need a specific dimension order)
    coords = {'x': x, 'y': y}
    # Create the xarray dataset
    ds = xr.DataArray(data, coords=coords, dims=['y', 'x'])
    # If you need to create a full Dataset (not just a DataArray), you can do:
    ds = ds.to_dataset(name='CGF_NDSI_Snow_Cover')
    return ds

def reproject_func(ds,var):
    ds = ds[var]
    ds.rio.write_crs("PROJCS[\"MODIS Sinusoidal\",GEOGCS[\"GCS_unnamed ellipse\",DATUM[\"D_unknown\",SPHEROID[\"Unknown\",6371007.181,0]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433]],PROJECTION[\"Sinusoidal\"],PARAMETER[\"false_easting\",0.0],PARAMETER[\"false_northing\",0.0],PARAMETER[\"central_meridian\",0.0],UNIT[\"Meter\",1.0]]", inplace=True)
#     ds = ds.rio.reproject(dom_ref.rio.crs)
#     ds.rio.reproject_match(dom_ref).plot(vmax=100)
    #ds = ds.sel(x=slice(x_lims[0],x_lims[1]),y=slice(y_lims[1],y_lims[0]))
    return ds

In [ ]:
var = 'CGF_NDSI_Snow_Cover'

date_list = []
ds_concat = []
for fCount,file in enumerate(files):
    print(file)
    
    try:
        ds = xr.open_dataset(file)
        print('dataset opened')
        
        ds = assign_and_grid(ds, file)
        print('assigned and gridded')
        
        #dom_ref = rxr.open_rasterio(file).isel(band=0).rio.write_crs(4326)
        #buffer = 5000
        #lim_test = dom_ref.rio.reproject("PROJCS[\"MODIS Sinusoidal\",GEOGCS[\"GCS_unnamed ellipse\",DATUM[\"D_unknown\",SPHEROID[\"Unknown\",6371007.181,0]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433]],PROJECTION[\"Sinusoidal\"],PARAMETER[\"false_easting\",0.0],PARAMETER[\"false_northing\",0.0],PARAMETER[\"central_meridian\",0.0],UNIT[\"Meter\",1.0]]")
        #x_lims = [lim_test.x.mean().values-buffer,lim_test.x.mean().values+buffer]
        #y_lims = [lim_test.y.mean().values-buffer,lim_test.y.mean().values+buffer]
        ds = reproject_func(ds,var)
        print('reprojected')
        
        ds = ds.where(ds <= 100)
        print('subsetted')

        if len(ds.values[~np.isnan(ds.values)]) > 0:
            print(ds.shape)
#             fg,ax = plt.subplots()
#             ds.plot(ax=ax,vmin=0,vmax=100)
#             ax.set_title(fCount)

            subsamp = file.full_name.split('/')[-1].split('.')[1]
            date_list.append(np.datetime64(datetime.date(int(subsamp[1:5]),1,1)+datetime.timedelta(days=int(subsamp[5:9])-1)))
            ds_concat.append(ds)
    except:
        print('potentially corrupted file?')
      
#     if fCount == 1:
#         break

ds_concat = xr.concat(ds_concat,dim="time")
ds_concat['time'] = ('time', date_list)

In [ ]:
multi_tile = 1
if multi_tile:
    # Group by unique times
    grouped = ds_concat.groupby('time')

    # Apply nanmean to each group, keeping it as an xarray.DataArray
    ds_concat = grouped.map(
        lambda group: xr.DataArray(
            np.nanmean(group, axis=0),  # Compute nanmean along the time axis
            dims=group.dims[1:],       # Keep the original spatial dimensions
            coords={dim: group.coords[dim] for dim in group.dims[1:]}  # Preserve coordinates
        )
    )

    # Check the resulting DataArray
    # print(ds_concat)
    ds_concat.rio.write_crs("PROJCS[\"MODIS Sinusoidal\",GEOGCS[\"GCS_unnamed ellipse\",DATUM[\"D_unknown\",SPHEROID[\"Unknown\",6371007.181,0]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433]],PROJECTION[\"Sinusoidal\"],PARAMETER[\"false_easting\",0.0],PARAMETER[\"false_northing\",0.0],PARAMETER[\"central_meridian\",0.0],UNIT[\"Meter\",1.0]]", inplace=True)

In [ ]:
ds_concat.mean(dim=('x','y')).plot()

In [ ]:
fig, ax = plt.subplots()
ds_concat.mean(dim='time').rio.reproject(4326).plot(ax=ax, vmin=0, vmax=100, cmap="Blues_r",
                                                    cbar_kwargs={'label': "Snow cover fraction [%]"})
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"VIIRS snow cover, 10-day mean")
ax.set_xlim(-180, -140)
ax.set_ylim(60, 70)
#cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.Esri.WorldImagery, zoom=8)

#ds_concat.mean(dim='time').plot()